In [1]:
import pandas as pd

df = pd.read_csv('../resources/movetimes3481887positions-withMoveNumber-withoutPieces.csv')
df.head(10)

,move_number,initial_time,increment,time_remaining_for_this_position,time_opponent_took,number_of_legal_moves,in_check,time_taken_for_this_position
0,2,600,0,600.03,3.60,28,False,1.28
1,3,600,0,596.43,1.28,31,False,3.52
2,3,600,0,598.75,3.52,32,False,1.44
3,4,600,0,592.91,1.44,31,False,3.28
4,4,600,0,597.31,3.28,41,False,1.36
5,5,600,0,589.63,1.36,29,False,6.32
6,5,600,0,595.95,6.32,43,False,1.44
7,6,600,0,583.31,1.44,32,False,10.16
8,6,600,0,594.51,10.16,47,False,2.00
9,7,600,0,573.15,2.00,33,False,13.76


In [2]:
from sklearn.model_selection import train_test_split

nontest, test = train_test_split(df, test_size = 0.1, random_state = 42)

train, val = train_test_split(nontest, test_size = 0.1, random_state = 42)
# given the size of the datatest, cross-validation is unnecessary to maintain a large training dataset.

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2820328 entries, 744012 to 3336675
Data columns (total 8 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   move_number                       int64  
 1   initial_time                      int64  
 2   increment                         int64  
 3   time_remaining_for_this_position  float64
 4   time_opponent_took                float64
 5   number_of_legal_moves             int64  
 6   in_check                          bool   
 7   time_taken_for_this_position      float64
dtypes: bool(1), float64(3), int64(4)
memory usage: 174.8 MB


In [4]:
val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 313370 entries, 489006 to 962524
Data columns (total 8 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   move_number                       313370 non-null  int64  
 1   initial_time                      313370 non-null  int64  
 2   increment                         313370 non-null  int64  
 3   time_remaining_for_this_position  313370 non-null  float64
 4   time_opponent_took                313370 non-null  float64
 5   number_of_legal_moves             313370 non-null  int64  
 6   in_check                          313370 non-null  bool   
 7   time_taken_for_this_position      313370 non-null  float64
dtypes: bool(1), float64(3), int64(4)
memory usage: 19.4 MB


In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 348189 entries, 3363431 to 298423
Data columns (total 8 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   move_number                       348189 non-null  int64  
 1   initial_time                      348189 non-null  int64  
 2   increment                         348189 non-null  int64  
 3   time_remaining_for_this_position  348189 non-null  float64
 4   time_opponent_took                348189 non-null  float64
 5   number_of_legal_moves             348189 non-null  int64  
 6   in_check                          348189 non-null  bool   
 7   time_taken_for_this_position      348189 non-null  float64
dtypes: bool(1), float64(3), int64(4)
memory usage: 21.6 MB


In [6]:
from sklearn.dummy import DummyRegressor
from sklearn import linear_model

dummy_regression_mean = DummyRegressor(strategy="mean")
dummy_regression_median = DummyRegressor(strategy="median")
linear_regression = linear_model.LinearRegression()

# 'time_taken_for_this_position' is the target feature, so we'll take it out and put it in a variables called y
def X_y_split(dataframe):
    X = dataframe.drop('time_taken_for_this_position', axis=1)
    y = dataframe['time_taken_for_this_position']
    return (X, y)

train_X, train_y = X_y_split(train)
val_X, val_y = X_y_split(val)

In [7]:
from sklearn.metrics import mean_squared_error

dummy_regression_mean.fit(train_X, train_y)
dummy_regression_median.fit(train_X, train_y)
linear_regression.fit(train_X, train_y)

print('dummy_regression_mean score on validation set:', mean_squared_error(val_y, dummy_regression_mean.predict(val_X)))
print('dummy_regression_median score on validation set:', mean_squared_error(val_y, dummy_regression_median.predict(val_X)))
print('linear_regression score on validation set:', mean_squared_error(val_y, linear_regression.predict(val_X)))
# lower mean_squared_error is better

dummy_regression_mean score on validation set: 73.32519935919755
dummy_regression_median score on validation set: 80.64002901937006
linear_regression score on validation set: 56.96629619421951


In [8]:
print('features and their linear regression coefficients')
coefficients = linear_regression.coef_
index = 0
for column in train_X:
    print(column, coefficients[index])
    index += 1

print('intercept:', linear_regression.intercept_)

features and their linear regression coefficients
move_number -0.04861498965587232
initial_time 0.02147350832229266
increment 0.39376505329200195
time_remaining_for_this_position -0.014886221184199772
time_opponent_took 0.19585766034746735
number_of_legal_moves 0.10686574532163687
in_check 1.0555790873368065
intercept: -1.3088811794857103


In [9]:
from skrub import TableReport
TableReport(df)

Processing column   8 / 8


<TableReport: use .open() to display>

In [10]:
# To check the distribution of predictions from our first linear regression model
TableReport(pd.DataFrame(linear_regression.predict(val_X)))

Processing column   1 / 1


<TableReport: use .open() to display>

In [11]:
# We can see that the median value of our target (time_taken_for_this_position in df) is 2.24 seconds,
# while the mean is 4.96 seconds (quite different). This shows the impact of outliers. 

# Our resulting linear regression model outputs a mean of 4.96 seconds but has a median of 4.07 seconds
# So our model has been impacted by the outliers.

# Thus, we will remove outliers from our training data (but not validation or test)
# as we don't want our model to output move times larger than is common just because the mean is skewed by outliers.

In [12]:
Q1 = train_y.quantile(0.25)
Q3 = train_y.quantile(0.75)
IQR = Q3 - Q1

mask = train_y <= (Q3 + 1.5 * IQR)
train_X = train_X[mask]
train_y = train_y[mask]


# Now we'll also remove samples where time_opponent_took are outliers. 
# This is because this is a very similar kind of feature to the target.
Q1 = train_X['time_opponent_took'].quantile(0.25)
Q3 = train_X['time_opponent_took'].quantile(0.75)
IQR = Q3 - Q1

mask = train_X['time_opponent_took'] <= (Q3 + 1.5 * IQR)
train_X = train_X[mask]
train_y = train_y[mask]


In [13]:
TableReport(train_X)

Processing column   7 / 7


<TableReport: use .open() to display>

In [14]:
TableReport(train_y)

Processing column   1 / 1


<TableReport: use .open() to display>

In [15]:
linear_regression.fit(train_X, train_y)
print('linear_regression score on validation set:', mean_squared_error(val_y, linear_regression.predict(val_X)))
# The score will be worse (higher error) as the validation data has outliers included, but we don't want to model outliers.

# This model gets chosen for implementation at the poster fair,
# so we'll be evaluating it on the test set here for it's final performance evaluation even though there's more to this notebook.
test_X, test_y = X_y_split(test)

chosen_model_test_performance = mean_squared_error(test_y, linear_regression.predict(test_X))

linear_regression score on validation set: 63.00082677336714


In [16]:
print('features and their linear regression coefficients')
coefficients = linear_regression.coef_
index = 0
for column in train_X:
    print(column, coefficients[index])
    index += 1

print('intercept:', linear_regression.intercept_)

features and their linear regression coefficients
move_number -0.00824053413451228
initial_time 0.0058190821748006
increment 0.06294285244760271
time_remaining_for_this_position -0.003140769331324015
time_opponent_took 0.26757500963761904
number_of_legal_moves 0.038792908621827236
in_check 0.5525076768512188
intercept: 0.2430456998921584


In [17]:
# To check the distribution of predictions from our outlier-removed linear regression model
TableReport(pd.DataFrame(linear_regression.predict(val_X)))

Processing column   1 / 1


<TableReport: use .open() to display>

In [18]:
# The distribution of predicted move times is looking better after removing outliers.
# There are also no longer a significant amount of negative values.
# Median is now 2.76 seconds, closer to actual median move times.

# Let's try using domain knowledge to create some engineered features that might improve the regression model.

In [19]:
import numpy as np
import skrub

# A feature to represent people hurrying up when they're low on time. 
# This feature doesn't get very large when time_remaining_for_this_position is large, but does quickly approach 0 
# when time_remaining_for_this_position approaches 0.
df['log_of_one_plus_time_remaining'] = np.log(1 + df['time_remaining_for_this_position'])

# Features to represent players no longer being in familiar territory (going "out of book"). 
# Multiple features will be made with different turning points to see which move number people actually tend to go out-of-book
df['out_of_book_depth_4'] = 1 / (1 + np.exp(4 - df['move_number']))
df['out_of_book_depth_5'] = 1 / (1 + np.exp(5 - df['move_number']))
df['out_of_book_depth_6'] = 1 / (1 + np.exp(6 - df['move_number']))
df['out_of_book_depth_7'] = 1 / (1 + np.exp(7 - df['move_number']))
df['out_of_book_depth_8'] = 1 / (1 + np.exp(8 - df['move_number']))
df['out_of_book_depth_9'] = 1 / (1 + np.exp(9 - df['move_number']))

target = 'time_taken_for_this_position'
associations = skrub.column_associations(df)

target_associations = associations[
    (associations['left_column_name'] == target) | 
    (associations['right_column_name'] == target)
]

In [20]:
target_associations.sort_values('pearson_corr', ascending=False)

,left_column_name,left_column_idx,right_column_name,right_column_idx,cramer_v,pearson_corr
55,initial_time,1,time_taken_for_this_position,7,0.054022,0.377734
51,time_opponent_took,4,time_taken_for_this_position,7,0.073236,0.344006
57,time_taken_for_this_position,7,log_of_one_plus_time_remaining,8,0.040813,0.320549
50,time_remaining_for_this_position,3,time_taken_for_this_position,7,0.077884,0.319203
58,increment,2,time_taken_for_this_position,7,0.027695,0.259601
71,number_of_legal_moves,5,time_taken_for_this_position,7,0.005171,0.139693
76,time_taken_for_this_position,7,out_of_book_depth_6,11,0.004229,0.094256
75,time_taken_for_this_position,7,out_of_book_depth_7,12,0.004273,0.093334
77,time_taken_for_this_position,7,out_of_book_depth_5,10,0.003999,0.092791
74,time_taken_for_this_position,7,out_of_book_depth_8,13,0.004345,0.090178


In [21]:
# As out_of_book_depth_6 has the best pearson_corr with the target feature, 
# and log_of_one_plus_time_remaining correlates well with the target feature,
# these two engineered features will be added to our linear regression classifier

In [22]:
# "out_of_book_depth_6" is a poor name. Using "opening_unfamiliarity" as the name instead.

def use_engineered_features(dataframe):
    dataframe['log_of_one_plus_time_remaining'] = np.log(1 + dataframe['time_remaining_for_this_position'])
    dataframe['opening_unfamiliarity'] = 1 / (1 + np.exp(6 - dataframe['move_number']))  
    return dataframe.drop(columns=['move_number', 'time_remaining_for_this_position'])

train_X = use_engineered_features(train_X)
val_X = use_engineered_features(val_X)

In [23]:
# To check that we've replace our old features with the engineered ones
train_X.head(10)

,initial_time,increment,time_opponent_took,number_of_legal_moves,in_check,log_of_one_plus_time_remaining,opening_unfamiliarity
696158,180,2,0.64,32,False,5.202192,0.047426
1506035,180,0,0.72,36,False,5.142307,0.731059
2722154,300,0,2.08,39,False,5.648657,0.982014
952210,60,2,1.52,35,False,4.056123,0.500000
453056,180,0,3.76,35,False,5.122952,0.952574
3127615,300,0,3.60,50,False,4.992268,1.000000
967834,600,0,1.92,22,False,5.892279,1.000000
1238127,180,0,2.32,32,False,2.811208,1.000000
2075708,60,0,1.04,43,False,4.060271,0.500000
2240494,60,0,0.00,26,False,2.359910,1.000000


In [24]:
linear_regression.fit(train_X, train_y)
print('linear_regression score on validation set:', mean_squared_error(val_y, linear_regression.predict(val_X)))

linear_regression score on validation set: 63.386359327146444


In [25]:
print('features and their linear regression coefficients')
coefficients = linear_regression.coef_
index = 0
for column in train_X:
    print(column, coefficients[index])
    index += 1

print('intercept:', linear_regression.intercept_)

features and their linear regression coefficients
initial_time 0.0016722979731779203
increment 0.01987816105375334
time_opponent_took 0.21683048255381254
number_of_legal_moves 0.026157947543738597
in_check 0.27861495972340555
log_of_one_plus_time_remaining 0.4679758223955675
opening_unfamiliarity 1.4766204701671053
intercept: -2.2769312079260655


In [26]:
# To check the distribution of this engineered-features regression model
TableReport(pd.DataFrame(linear_regression.predict(val_X)))

Processing column   1 / 1


<TableReport: use .open() to display>

In [27]:
# Using the engineered features reduced validation performance (63.39 > 63.00)
# So the model used for the realistic move times feature will be a simple linear regression one with the following coefficients:

# move_number -0.00824053413451228
# initial_time 0.0058190821748006
# increment 0.06294285244760271
# time_remaining_for_this_position -0.003140769331324015
# time_opponent_took 0.26757500963761904
# number_of_legal_moves 0.038792908621827236
# in_check 0.5525076768512188
# intercept: 0.2430456998921584

print("Chosen model's mean squared error on test data:", chosen_model_test_performance)

Chosen model's mean squared error on test data: 64.42041733404085


In [1]:
#### UPDATE ####

# After local testing, this model is unsatisfactory in practice. It doesn't feel realistic enough.
# Spends too long in the opening and isn't playing re-captures quickly.
# This model WON'T be used for the poster fair